# Lab 5 - Streaming REPL with a Live Cost Meter

**Section 7 · The Event Stream**  ·  **Estimated time:** 20–25 min  ·  **Estimated cost:** a few cents

This is a **Jupyter Notebook**. Run it top to bottom in **Udemy Labs** (nothing to install) or on your own machine. Read the note above each cell, then run the cell with Shift+Enter.

> **Cost note:** the meter prints a list-price estimate from `session.usage` plus Managed Agents active runtime. It is not an authoritative invoice. Check Claude Console billing for final cost.

## Goal
Build a tiny streaming console: chat with one persistent session, watch the agent's text and tool use stream in live, and see a running cost ticker after every turn. You will learn to read the streaming event loop, surface tool calls in band, and turn `session.usage` into a dollar figure on your own UI.

## Prerequisites
- the shared uv kernel `Managed Agents Labs (.venv)`
- An API key you can paste into the setup cell below. The notebook stores it as `ANTHROPIC_API_KEY` for this kernel session only.

In [ ]:
# Verify that this notebook is using the uv-managed lab kernel.
import sys
from pathlib import Path

EXPECTED_KERNEL = "Managed Agents Labs (.venv)"
python_exe = Path(sys.executable)
python_prefix = Path(sys.prefix)

if ".venv" not in python_exe.parts and ".venv" not in python_prefix.parts:
    raise RuntimeError(
        f"Select the Jupyter kernel {EXPECTED_KERNEL!r} before running this notebook. "
        "From the repo root, run ./setup_uv.sh once to create and register it."
    )

print("Using uv environment:", python_prefix)

In [ ]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

print("ANTHROPIC_API_KEY configured for this notebook session.")

In [ ]:
import os, sys
from anthropic import Anthropic

# Udemy Labs note: the previous cell configures ANTHROPIC_API_KEY for this session.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("MODEL", "claude-haiku-4-5-20251001")  # course default; swap as models update
client = Anthropic()
print("SDK ready, model:", MODEL)

### Step 1 - Create the agent, environment, and session
Create them once; the whole REPL talks to this single session.

In [ ]:
agent = client.beta.agents.create(
    name="Streaming Console",
    model=MODEL,
    system="You are a concise console assistant. Use the bash tool to inspect the workspace when asked. Keep replies short.",
    tools=[{"type": "agent_toolset_20260401"}],
    betas=BETAS,
)
env = client.beta.environments.create(
    name="streaming-repl",
    config={"type": "cloud", "networking": {"type": "unrestricted"}},
    betas=BETAS,
)
session = client.beta.sessions.create(
    agent=agent.id, environment_id=env.id, title="Lab 5 streaming REPL", betas=BETAS,
)
print("session.id =", session.id)

### Step 2 - A cost helper
Re-fetch the session after each idle and print the shared list-price estimate from cumulative `session.usage` plus Managed Agents active runtime.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost


def show_cost(session_id):
    print_sessions_cost(client, [session_id], MODEL, betas=BETAS)


### Step 3 - Send one turn and stream it
In a notebook we send a turn and consume the stream in a function (no blocking `input()` loop). Run this cell, then change `prompt` and run again to continue the same session.

In [ ]:
def turn(prompt):
    with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
        client.beta.sessions.events.send(
            session.id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": prompt}]}],
            betas=BETAS,
        )
        sys.stdout.write("agent> "); sys.stdout.flush()
        for event in stream:
            if event.type == "agent.message":
                for block in event.content:
                    if block.type == "text":
                        sys.stdout.write(block.text); sys.stdout.flush()
            elif event.type == "agent.tool_use":
                sys.stdout.write(f"\n  [tool: {event.name}]\n  "); sys.stdout.flush()
            elif event.type == "session.status_idle":
                show_cost(session.id); break

turn("What's in /workspace?")

In [ ]:
# Run another turn on the same session - the cost total climbs.
turn("Read /etc/os-release and summarize it in one line.")

### Cost estimate
Re-fetch the session id(s), print one list-price estimate per session, then print the total across all session ids. The estimate uses cumulative token usage plus Managed Agents active runtime; treat it as a course meter, not an invoice.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost

print_sessions_cost(client, [session.id], MODEL, betas=BETAS)


## Expected output
- A `session.id` line.
- Agent text streaming after `agent> `, with a `[tool: bash]` marker each time it reaches for a tool.
- An `Estimated lab cost` block after each turn, climbing as usage and active runtime accumulate.

## Troubleshooting
- **Cost always $0.00 / `usage` is None** → read usage only after `session.status_idle`, and re-fetch with `client.beta.sessions.retrieve(...)` (the create-time object is stale).
- **Stream hangs** → SSE is one long connection; a dropped network closes it. Re-run the `turn(...)` cell to open a fresh stream. To catch up an in-flight turn, `client.beta.sessions.events.list(session.id, betas=BETAS)`.
- **The number looks wrong** → it is a list-price estimate, not an invoice. Source final numbers from Claude Console billing. Cache counters come from `session.usage.cache_read_input_tokens` and `session.usage.cache_creation_input_tokens`.

## Bonus (optional): drive this lab with Claude Code

Not required - the notebook above is the whole lab. If you want to try **agentic engineering**, open this folder in Claude Code and paste:

> "Write a small streaming console for the Claude Managed Agents beta (`betas=['managed-agents-2026-04-01']`, all calls on `client.beta.*`). Create an agent on `claude-haiku-4-5-20251001` with `{'type': 'agent_toolset_20260401'}`, a plain cloud environment, and one session. Send a turn, open `client.beta.sessions.events.stream(...)`, print `agent.message` text live and every `agent.tool_use` as `[tool: <name>]`, and on `session.status_idle` print a running cost from `session.usage`."

Then ask it to reuse `../shared/cost_meter.py` for the running estimate. Compare what Claude Code writes to the cells above.

## Stretch
- Color the output with `rich` (tool markers cyan, cost line green).
- Keep a `collections.Counter` of tool names and print `bash x3, edit x1` alongside the cost.
- Persist `session.id` and reattach on the next run instead of creating a new session.